# Crypto Sentinel Pro — Self-Learning GitHub Edition (Fixed)

A key-free market-analysis and paper-trading agent for Kaggle. It scans BTC, ETH and SOL, accumulates candle history, retrains chronological per-asset models, checks a learning-quality gate, and only then permits paper trades.

### Corrected persistent-learning cycle
1. Pull prior runtime data from `fahadumrani/trader_agent` using the repository's detected default branch.
2. Normalize timestamps, merge fresh candles with history, and remove duplicates.
3. Train on completed future outcomes only; use the latest unlabeled row strictly for inference.
4. Validate chronologically and require minimum rows, accuracy and AUC before trading.
5. Select the strongest **learned-ready** asset instead of blocking on an unready leader.
6. Save candles, model records, signals, trades, state, equity and reports to GitHub every 30 minutes and at shutdown.

**Safety boundary:** paper trading only. No model guarantees profit. Store `GITHUB_TOKEN` only in Kaggle Secrets; never paste it into notebook code.


In [ ]:
# Kaggle: enable Internet, then Run All
!pip -q install yfinance


In [ ]:
import os, json, time, warnings, base64
from urllib.parse import quote
import requests
from datetime import datetime, timezone

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import yfinance as yf

warnings.filterwarnings("ignore")
plt.style.use("seaborn-v0_8-whitegrid")

# ==================== USER-SAFE DEFAULTS ====================
SYMBOLS = ["BTC-USD", "ETH-USD", "SOL-USD"]
STARTING_CASH = 5.00
SESSION_HOURS = 12
POLL_SECONDS = 300
DATA_PERIOD = "7d"
DATA_INTERVAL = "5m"

FEE_RATE = 0.0010
SLIPPAGE_RATE = 0.0005
RISK_PER_TRADE = 0.01
MAX_POSITION_FRACTION = 0.80
MAX_DAILY_LOSS_FRACTION = 0.03
MAX_TRADES_PER_DAY = 8
MAX_CONSECUTIVE_LOSSES = 3
COOLDOWN_CANDLES = 3
MAX_HOLD_CANDLES = 24
ENTRY_SCORE = 68
EXIT_SCORE = 42
MIN_MODEL_ACCURACY = 0.50
MIN_MODEL_AUC = 0.51
MIN_LEARNING_ROWS = 500
GITHUB_SYNC_MINUTES = 30
TARGET_HORIZON_BARS = 3
MIN_TARGET_EDGE = 0.0005
MAX_HISTORY_ROWS = 25000
CLOSE_POSITION_AT_SESSION_END = True

BASE_DIR = "/kaggle/working/trader_agent_runtime" if os.path.exists("/kaggle/working") else "./trader_agent_runtime"
os.makedirs(BASE_DIR, exist_ok=True)
GITHUB_OWNER = "fahadumrani"
GITHUB_REPO = "trader_agent"
GITHUB_BRANCH = "main"
GITHUB_RUNTIME_PREFIX = "runtime"
STATE_FILE = os.path.join(BASE_DIR, "sentinel_state.json")
TRADES_FILE = os.path.join(BASE_DIR, "sentinel_trades.csv")
SIGNALS_FILE = os.path.join(BASE_DIR, "sentinel_signals.csv")
EQUITY_FILE = os.path.join(BASE_DIR, "sentinel_equity.csv")
REPORT_PNG = os.path.join(BASE_DIR, "sentinel_report.png")

print("Crypto Sentinel Pro configured")
print("Universe:", ", ".join(SYMBOLS), "| Virtual capital: $", STARTING_CASH)


In [ ]:
# ==================== SECURE GITHUB PERSISTENCE ====================
def get_github_token():
    token=os.environ.get("GITHUB_TOKEN","").strip()
    if token: return token
    try:
        from kaggle_secrets import UserSecretsClient
        return (UserSecretsClient().get_secret("GITHUB_TOKEN") or "").strip()
    except Exception:
        return ""

GITHUB_TOKEN=get_github_token()
GITHUB_REPO_API=f"https://api.github.com/repos/{GITHUB_OWNER}/{GITHUB_REPO}"
GITHUB_CONTENTS_API=GITHUB_REPO_API+"/contents"
ACTIVE_GITHUB_BRANCH=GITHUB_BRANCH


def github_headers(raw=False):
    accept="application/vnd.github.raw+json" if raw else "application/vnd.github+json"
    headers={"Accept":accept,"X-GitHub-Api-Version":"2022-11-28"}
    if GITHUB_TOKEN: headers["Authorization"]="Bearer "+GITHUB_TOKEN
    return headers


def github_resolve_branch():
    global ACTIVE_GITHUB_BRANCH
    try:
        r=requests.get(GITHUB_REPO_API,headers=github_headers(),timeout=30)
        if r.ok:
            ACTIVE_GITHUB_BRANCH=r.json().get("default_branch") or GITHUB_BRANCH
    except Exception as exc:
        print("GitHub branch detection warning:",exc)
    return ACTIVE_GITHUB_BRANCH


def _request_with_retry(method,url,**kwargs):
    last=None
    for attempt in range(3):
        try:
            response=requests.request(method,url,**kwargs)
            if response.status_code not in (429,500,502,503,504): return response
            last=RuntimeError(f"GitHub temporary HTTP {response.status_code}")
        except Exception as exc:
            last=exc
        time.sleep(2**attempt)
    raise last or RuntimeError("GitHub request failed")


def github_get(repo_path):
    encoded=quote(repo_path,safe="/")
    url=f"{GITHUB_CONTENTS_API}/{encoded}"
    r=_request_with_retry("GET",url,headers=github_headers(),params={"ref":ACTIVE_GITHUB_BRANCH},timeout=30)
    if r.status_code==404: return None,None
    r.raise_for_status(); obj=r.json(); sha=obj.get("sha")
    content=obj.get("content")
    if content and obj.get("encoding")=="base64":
        return base64.b64decode(content),sha
    download_url=obj.get("download_url")
    if download_url:
        raw=_request_with_retry("GET",download_url,headers=github_headers(raw=True),timeout=60)
    else:
        raw=_request_with_retry("GET",url,headers=github_headers(raw=True),params={"ref":ACTIVE_GITHUB_BRANCH},timeout=60)
    raw.raise_for_status(); return raw.content,sha


def github_put(repo_path,payload,message):
    if not GITHUB_TOKEN: return False
    if len(payload)>90*1024*1024:
        raise ValueError(f"Refusing GitHub file over 90 MB: {repo_path}")
    _,sha=github_get(repo_path)
    body={"message":message,"content":base64.b64encode(payload).decode(),"branch":ACTIVE_GITHUB_BRANCH}
    if sha: body["sha"]=sha
    url=f"{GITHUB_CONTENTS_API}/{quote(repo_path,safe='/')}"
    r=_request_with_retry("PUT",url,headers=github_headers(),json=body,timeout=90)
    r.raise_for_status(); return True


def pull_previous_learning():
    github_resolve_branch()
    try:
        raw,_=github_get(f"{GITHUB_RUNTIME_PREFIX}/manifest.json")
        if not raw:
            print("No prior GitHub learning manifest; starting with local/fresh history."); return
        manifest=json.loads(raw.decode("utf-8")); pulled=0
        for rel in manifest.get("files",[]):
            data,_=github_get(f"{GITHUB_RUNTIME_PREFIX}/{rel}")
            if data is not None:
                path=os.path.join(BASE_DIR,rel); os.makedirs(os.path.dirname(path),exist_ok=True)
                with open(path,"wb") as handle: handle.write(data)
                pulled+=1
        print(f"Pulled {pulled} prior learning files from GitHub ({ACTIVE_GITHUB_BRANCH}).")
    except Exception as exc:
        print("GitHub pull warning; continuing locally:",exc)


def sync_all_to_github():
    if not GITHUB_TOKEN:
        print("GitHub sync skipped: add GITHUB_TOKEN in Kaggle Secrets."); return False
    github_resolve_branch(); files=[]; failed=[]
    for root,_,names in os.walk(BASE_DIR):
        for name in sorted(names):
            if not name.endswith((".csv",".json",".png")): continue
            full=os.path.join(root,name); rel=os.path.relpath(full,BASE_DIR).replace(os.sep,"/")
            try:
                with open(full,"rb") as handle: payload=handle.read()
                github_put(f"{GITHUB_RUNTIME_PREFIX}/{rel}",payload,f"Auto-save agent data: {rel}")
                files.append(rel)
            except Exception as exc:
                failed.append(rel); print(f"GitHub sync warning for {rel}: {exc}")
    try:
        manifest=json.dumps({"updated_at":now_iso(),"branch":ACTIVE_GITHUB_BRANCH,
                             "files":sorted(set(files+failed)),"failed_this_sync":failed},indent=2).encode()
        github_put(f"{GITHUB_RUNTIME_PREFIX}/manifest.json",manifest,"Update agent learning manifest")
    except Exception as exc:
        print("GitHub manifest warning:",exc); return False
    print(f"GitHub sync complete: {len(files)} uploaded, {len(failed)} failed.")
    return not failed


In [ ]:
# ==================== DATA & SELF-LEARNING FEATURES ====================
def now_iso():
    return datetime.now(timezone.utc).isoformat()


def append_csv(path,row):
    os.makedirs(os.path.dirname(path) or ".",exist_ok=True)
    pd.DataFrame([row]).to_csv(path,mode="a",header=not os.path.exists(path),index=False)


FEATURES=["ret1","ret3","ema_gap","rsi","macd_hist","bb_pos","atr_pct","adx","vol_z","breakout20"]


def download_5m(symbol):
    fresh=yf.download(symbol,period=DATA_PERIOD,interval=DATA_INTERVAL,
                      auto_adjust=True,progress=False,threads=False)
    if fresh is None or fresh.empty: raise RuntimeError(f"No fresh data for {symbol}")
    if isinstance(fresh.columns,pd.MultiIndex): fresh.columns=fresh.columns.get_level_values(0)
    required=["Open","High","Low","Close","Volume"]
    missing=[c for c in required if c not in fresh.columns]
    if missing: raise RuntimeError(f"{symbol} missing columns: {missing}")
    fresh=fresh[required].dropna().copy()
    fresh.index=pd.to_datetime(fresh.index,utc=True,errors="coerce")
    fresh=fresh[~fresh.index.isna()].iloc[:-1]
    history_path=os.path.join(BASE_DIR,f"market_{symbol.replace('-', '_')}_5m.csv")
    frames=[]
    if os.path.exists(history_path):
        try:
            old=pd.read_csv(history_path,index_col=0)
            old.index=pd.to_datetime(old.index,utc=True,errors="coerce")
            old=old[~old.index.isna()]
            for col in required: old[col]=pd.to_numeric(old[col],errors="coerce")
            frames.append(old[required].dropna())
        except Exception as exc:
            print(f"Ignoring damaged local history for {symbol}: {exc}")
    frames.append(fresh)
    merged=pd.concat(frames).sort_index()
    merged=merged[~merged.index.duplicated(keep="last")].tail(MAX_HISTORY_ROWS)
    merged.to_csv(history_path)
    if len(merged)<250: raise RuntimeError(f"Insufficient data for {symbol}: {len(merged)} rows")
    return merged


def resample_ohlcv(df,rule):
    return df.resample(rule,label="right",closed="right").agg({
        "Open":"first","High":"max","Low":"min","Close":"last","Volume":"sum"
    }).dropna()


def indicators(df):
    x=df.copy(); c,h,l,v=x.Close,x.High,x.Low,x.Volume
    x["ret1"]=c.pct_change(); x["ret3"]=c.pct_change(3)
    x["ema9"]=c.ewm(span=9,adjust=False).mean(); x["ema21"]=c.ewm(span=21,adjust=False).mean()
    x["ema50"]=c.ewm(span=50,adjust=False).mean(); x["ema_gap"]=x.ema9/x.ema21-1
    delta=c.diff(); gain=delta.clip(lower=0); loss=-delta.clip(upper=0)
    avg_gain=gain.ewm(alpha=1/14,adjust=False).mean(); avg_loss=loss.ewm(alpha=1/14,adjust=False).mean()
    x["rsi"]=100-100/(1+avg_gain/avg_loss.replace(0,np.nan))
    macd=c.ewm(span=12,adjust=False).mean()-c.ewm(span=26,adjust=False).mean()
    x["macd"]=macd; x["macd_signal"]=macd.ewm(span=9,adjust=False).mean()
    x["macd_hist"]=(x.macd-x.macd_signal)/c
    mid=c.rolling(20).mean(); sd=c.rolling(20).std()
    x["bb_mid"]=mid; x["bb_upper"]=mid+2*sd; x["bb_lower"]=mid-2*sd
    x["bb_pos"]=(c-x.bb_lower)/(x.bb_upper-x.bb_lower).replace(0,np.nan)
    previous=c.shift(1)
    true_range=pd.concat([(h-l),(h-previous).abs(),(l-previous).abs()],axis=1).max(axis=1)
    x["atr"]=true_range.ewm(alpha=1/14,adjust=False).mean(); x["atr_pct"]=x.atr/c
    up=h.diff(); down=-l.diff()
    plus_dm=up.where((up>down)&(up>0),0.0); minus_dm=down.where((down>up)&(down>0),0.0)
    atr14=true_range.ewm(alpha=1/14,adjust=False).mean()
    x["plus_di"]=100*plus_dm.ewm(alpha=1/14,adjust=False).mean()/atr14
    x["minus_di"]=100*minus_dm.ewm(alpha=1/14,adjust=False).mean()/atr14
    dx=100*(x.plus_di-x.minus_di).abs()/(x.plus_di+x.minus_di).replace(0,np.nan)
    x["adx"]=dx.ewm(alpha=1/14,adjust=False).mean()
    x["vol_z"]=(v-v.rolling(30).mean())/v.rolling(30).std().replace(0,np.nan)
    x["breakout20"]=c/h.rolling(20).max().shift(1)-1
    x["future_return"]=c.shift(-TARGET_HORIZON_BARS)/c-1
    threshold=2*(FEE_RATE+SLIPPAGE_RATE)+MIN_TARGET_EDGE
    x["target"]=np.where(x.future_return.notna(),(x.future_return>threshold).astype(float),np.nan)
    x=x.replace([np.inf,-np.inf],np.nan)
    return x.dropna(subset=FEATURES+["atr","ema9","ema21","ema50","plus_di","minus_di"])


def _sigmoid(z): return 1/(1+np.exp(-np.clip(z,-30,30)))


def _fit_logistic(X,y,steps=350,lr=0.06,l2=0.02):
    mu=X.mean(axis=0); sigma=X.std(axis=0)+1e-9; Z=(X-mu)/sigma
    w=np.zeros(Z.shape[1]); b=0.0
    for _ in range(steps):
        p=_sigmoid(Z@w+b); w-=lr*((Z.T@(p-y))/len(y)+l2*w); b-=lr*np.mean(p-y)
    return w,b,mu,sigma


def _predict_logistic(X,model):
    w,b,mu,sigma=model; return _sigmoid(((X-mu)/sigma)@w+b)


def _auc_rank(y,p):
    y=np.asarray(y); p=np.asarray(p); positive=y==1; negative=y==0
    if positive.sum()==0 or negative.sum()==0: return 0.50
    ranks=pd.Series(p).rank(method="average").to_numpy()
    return float((ranks[positive].sum()-positive.sum()*(positive.sum()+1)/2)/(positive.sum()*negative.sum()))


def ml_probability(frame,symbol=None):
    inference=frame.dropna(subset=FEATURES).iloc[[-1]]
    labeled=frame.dropna(subset=FEATURES+["target"]).copy()
    rows=len(labeled)
    if rows<220 or labeled.target.nunique()<2:
        return 0.50,0.0,0.50,rows
    X=labeled[FEATURES].to_numpy(float); y=labeled.target.to_numpy(float)
    split=max(150,int(rows*0.80))
    if rows-split<30: return 0.50,0.0,0.50,rows
    validation_model=_fit_logistic(X[:split],y[:split])
    probabilities=_predict_logistic(X[split:],validation_model); actual=y[split:]
    accuracy=float(np.mean((probabilities>=0.5)==actual)); auc=_auc_rank(actual,probabilities)
    final_model=_fit_logistic(X,y)
    current_probability=float(_predict_logistic(inference[FEATURES].to_numpy(float),final_model)[0])
    if symbol:
        w,b,mu,sigma=final_model
        record={"symbol":symbol,"trained_at":now_iso(),"labeled_rows":rows,"horizon_bars":TARGET_HORIZON_BARS,
                "target_threshold":2*(FEE_RATE+SLIPPAGE_RATE)+MIN_TARGET_EDGE,"features":FEATURES,
                "weights":w.tolist(),"bias":float(b),"mean":mu.tolist(),"scale":sigma.tolist(),
                "validation_accuracy":accuracy,"validation_auc":auc,"latest_probability":current_probability}
        with open(os.path.join(BASE_DIR,f"model_{symbol.replace('-', '_')}.json"),"w") as handle:
            json.dump(record,handle,indent=2)
    return current_probability,accuracy,auc,rows


In [ ]:
# ==================== DECISION ENGINE ====================
def market_regime(hourly):
    row=hourly.iloc[-1]; slope=hourly.ema21.iloc[-1]/hourly.ema21.iloc[-4]-1
    if row.atr_pct>hourly.atr_pct.tail(60).quantile(0.85): return "high_volatility"
    if row.adx>=22 and slope>0 and row.Close>row.ema50: return "uptrend"
    if row.adx>=22 and slope<0: return "downtrend"
    return "range"


def score_symbol(symbol,raw):
    f5=indicators(raw); f15=indicators(resample_ohlcv(raw,"15min")); h1=indicators(resample_ohlcv(raw,"1h"))
    if min(len(f5),len(f15),len(h1))<4: raise RuntimeError(f"Insufficient indicator history for {symbol}")
    a,b,c=f5.iloc[-1],f15.iloc[-1],h1.iloc[-1]
    regime=market_regime(h1)
    ml_prob,ml_acc,ml_auc,learning_rows=ml_probability(f5,symbol)
    technical=0.0; reasons=[]
    if a.ema9>a.ema21: technical+=12; reasons.append("5m trend up")
    if b.ema9>b.ema21: technical+=13; reasons.append("15m confirms")
    if c.Close>c.ema50: technical+=10; reasons.append("1h above EMA50")
    if a.macd_hist>0: technical+=8; reasons.append("MACD positive")
    if 48<=a.rsi<=68: technical+=8; reasons.append("RSI healthy")
    elif a.rsi>76: technical-=8; reasons.append("RSI overheated")
    if a.plus_di>a.minus_di and a.adx>=18: technical+=8; reasons.append("ADX buyers")
    if a.vol_z>0.25: technical+=5; reasons.append("volume confirmation")
    if a.breakout20>0: technical+=6; reasons.append("20-bar breakout")
    if regime=="uptrend": technical+=10; reasons.append("uptrend regime")
    elif regime=="downtrend": technical-=18; reasons.append("downtrend regime")
    elif regime=="high_volatility": technical-=8; reasons.append("volatility penalty")
    technical=float(np.clip(technical,0,80))
    model_trusted=ml_acc>=MIN_MODEL_ACCURACY and ml_auc>=MIN_MODEL_AUC
    score=float(np.clip(technical+20*(ml_prob if model_trusted else 0.50),0,100))
    ready=learning_rows>=MIN_LEARNING_ROWS and model_trusted
    return {"symbol":symbol,"score":score,"price":float(a.Close),"atr":float(a.atr),"rsi":float(a.rsi),
            "regime":regime,"ml_probability":ml_prob,"ml_accuracy":ml_acc,"ml_auc":ml_auc,
            "learning_rows":learning_rows,"learned_ready":bool(ready),"reasons":"; ".join(reasons),
            "candle":str(f5.index[-1])}


def scan_market():
    ranked=[]; frames={}
    for symbol in SYMBOLS:
        try:
            raw=download_5m(symbol); frames[symbol]=raw; ranked.append(score_symbol(symbol,raw))
        except Exception as exc:
            print(f"{symbol} scan error: {exc}")
    if not ranked: raise RuntimeError("No symbols could be scanned. Check Kaggle Internet and data format.")
    ranked.sort(key=lambda item:item["score"],reverse=True)
    return ranked,frames


In [ ]:
# ==================== PORTFOLIO & RISK ENGINE ====================
def fresh_state():
    return {"cash":STARTING_CASH, "position":None, "peak_equity":STARTING_CASH,
            "realized_pnl":0.0, "trade_count":0, "trades_today":0,
            "day":datetime.now(timezone.utc).date().isoformat(),
            "day_start_equity":STARTING_CASH, "consecutive_losses":0,
            "cooldown":0, "last_candle":None, "locked":False}


def load_state():
    state=fresh_state()
    if os.path.exists(STATE_FILE):
        try:
            with open(STATE_FILE) as handle: saved=json.load(handle)
            state.update(saved)
        except Exception as exc:
            print("State recovery warning:",exc)
    return state


def save_state(state):
    tmp=STATE_FILE+".tmp"
    with open(tmp,"w") as f: json.dump(state,f,indent=2)
    os.replace(tmp,STATE_FILE)


def total_equity(state, prices):
    pos=state.get("position")
    return float(state["cash"] + (pos["qty"] * prices.get(pos["symbol"], pos["entry"]) if pos else 0))


def roll_day(state, eq):
    today=datetime.now(timezone.utc).date().isoformat()
    if state["day"] != today:
        state.update({"day":today,"day_start_equity":eq,"trades_today":0,
                      "consecutive_losses":0,"locked":False})


def risk_allows_entry(state, equity_now):
    daily_loss=(state["day_start_equity"]-equity_now)/max(state["day_start_equity"],1e-9)
    if daily_loss >= MAX_DAILY_LOSS_FRACTION: state["locked"]=True
    return (not state["locked"] and state["trades_today"] < MAX_TRADES_PER_DAY
            and state["consecutive_losses"] < MAX_CONSECUTIVE_LOSSES
            and state["cooldown"] <= 0 and state["position"] is None)


def enter(state, setup):
    entry=setup["price"]*(1+SLIPPAGE_RATE)
    stop_pct=float(np.clip(1.6*setup["atr"]/entry,0.008,0.025))
    stop=entry*(1-stop_pct); target=entry*(1+2.0*stop_pct)
    risk_dollars=max(0, total_equity(state,{setup["symbol"]:entry})*RISK_PER_TRADE)
    qty_by_risk=risk_dollars/max(entry-stop,1e-9)
    budget_cap=state["cash"]*MAX_POSITION_FRACTION
    qty=min(qty_by_risk, budget_cap/(entry*(1+FEE_RATE)))
    if qty<=0: return
    gross=qty*entry; fee=gross*FEE_RATE
    state["cash"]-=gross+fee
    state["position"]={"symbol":setup["symbol"],"qty":qty,"entry":entry,
        "entry_fee":fee,"stop":stop,"initial_stop":stop,"target":target,
        "highest":entry,"opened_at":now_iso(),"bars":0,"entry_score":setup["score"]}
    save_state(state)
    print(f"BUY {setup['symbol']} | score {setup['score']:.1f} | ${gross:.4f} | stop {stop:.2f} | target {target:.2f}")


def exit_position(state, price, reason):
    p=state["position"]
    fill=price*(1-SLIPPAGE_RATE); gross=p["qty"]*fill; fee=gross*FEE_RATE
    proceeds=gross-fee
    cost=p["qty"]*p["entry"]+p["entry_fee"]
    pnl=proceeds-cost
    state["cash"]+=proceeds; state["realized_pnl"]+=pnl
    state["trade_count"]+=1; state["trades_today"]+=1
    state["consecutive_losses"] = state["consecutive_losses"]+1 if pnl<0 else 0
    state["cooldown"]=COOLDOWN_CANDLES
    append_csv(TRADES_FILE,{"opened_at":p["opened_at"],"closed_at":now_iso(),
        "symbol":p["symbol"],"entry":p["entry"],"exit":fill,"qty":p["qty"],
        "entry_score":p["entry_score"],"pnl_usd":pnl,
        "return_pct":100*pnl/max(cost,1e-9),"reason":reason})
    print(f"SELL {p['symbol']} | {reason} | P/L ${pnl:.4f}")
    state["position"]=None
    save_state(state)


def manage_position(state, setup_map):
    p=state.get("position")
    if not p or p["symbol"] not in setup_map: return
    s=setup_map[p["symbol"]]; price=s["price"]
    p["bars"]+=1; p["highest"]=max(p["highest"],price)
    risk=p["entry"]-p["initial_stop"]
    if price >= p["entry"]+risk:
        p["stop"]=max(p["stop"],p["entry"]*(1+FEE_RATE+SLIPPAGE_RATE))
    if price >= p["entry"]+1.5*risk:
        p["stop"]=max(p["stop"],p["highest"]-1.4*s["atr"])
    reason=None
    if price<=p["stop"]: reason="adaptive_stop"
    elif price>=p["target"]: reason="take_profit"
    elif s["score"]<EXIT_SCORE: reason="ensemble_exit"
    elif s["regime"]=="downtrend": reason="regime_exit"
    elif p["bars"]>=MAX_HOLD_CANDLES: reason="time_exit"
    if reason: exit_position(state,price,reason)


In [ ]:
# ==================== AUTONOMOUS 12-HOUR LOOP ====================
def run_sentinel():
    pull_previous_learning()
    state=load_state(); end=time.time()+SESSION_HOURS*3600; last_prices={}; errors=0
    next_sync=time.time()+GITHUB_SYNC_MINUTES*60
    print(f"Starting self-learning PAPER session for up to {SESSION_HOURS} hours.\n")
    try:
        while time.time()<end:
            try:
                ranked,_=scan_market(); setup_map={x["symbol"]:x for x in ranked}
                last_prices={x["symbol"]:x["price"] for x in ranked}; leader=ranked[0]
                candle=leader["candle"]
                if state.get("last_candle")==candle:
                    if time.time()>=next_sync:
                        sync_all_to_github(); next_sync=time.time()+GITHUB_SYNC_MINUTES*60
                    time.sleep(min(POLL_SECONDS,max(1,end-time.time()))); continue

                for item in ranked: append_csv(SIGNALS_FILE,{"time":now_iso(),**item})
                equity_now=total_equity(state,last_prices); roll_day(state,equity_now)
                had_position=state.get("position") is not None
                if not had_position and state["cooldown"]>0: state["cooldown"]-=1
                manage_position(state,setup_map)
                equity_now=total_equity(state,last_prices)

                ready_candidates=[item for item in ranked if item["learned_ready"]]
                candidate=ready_candidates[0] if ready_candidates else None
                if candidate is None:
                    print("LEARNING GATE: no asset passed rows/accuracy/AUC; observation only.")
                elif risk_allows_entry(state,equity_now) and candidate["score"]>=ENTRY_SCORE and candidate["regime"]!="downtrend":
                    enter(state,candidate)

                equity_now=total_equity(state,last_prices); state["peak_equity"]=max(state["peak_equity"],equity_now)
                drawdown=(equity_now/state["peak_equity"]-1)*100
                append_csv(EQUITY_FILE,{"time":now_iso(),"equity":equity_now,"cash":state["cash"],
                    "drawdown_pct":drawdown,"leader":leader["symbol"],"leader_score":leader["score"],
                    "candidate":candidate["symbol"] if candidate else None,
                    "open_symbol":state["position"]["symbol"] if state["position"] else None})
                state["last_candle"]=candle; save_state(state); errors=0
                if time.time()>=next_sync:
                    sync_all_to_github(); next_sync=time.time()+GITHUB_SYNC_MINUTES*60
                print(f"{now_iso()} | leader {leader['symbol']} {leader['score']:.1f} | {leader['regime']} | equity ${equity_now:.4f}")
            except Exception as exc:
                errors+=1; print(f"Cycle error {errors}: {exc}")
                if errors>=5:
                    print("Five consecutive cycle errors; stopping safely with checkpoint."); break
            remaining=end-time.time()
            if remaining>0: time.sleep(min(POLL_SECONDS,remaining))
    except KeyboardInterrupt:
        print("Stopped manually; checkpoint preserved.")
    finally:
        position=state.get("position")
        if CLOSE_POSITION_AT_SESSION_END and position and position["symbol"] in last_prices:
            exit_position(state,last_prices[position["symbol"]],"session_end")
        save_state(state); final_report(state,last_prices)
        try: sync_all_to_github()
        except Exception as exc: print("Final GitHub sync warning:",exc)


def final_report(state,last_prices=None):
    last_prices=last_prices or {}; position=state.get("position")
    final_equity=total_equity(state,last_prices)
    pnl=final_equity-STARTING_CASH
    print("\n================ SENTINEL FINAL REPORT ================")
    print(f"Start: ${STARTING_CASH:.4f} | End equity: ${final_equity:.4f} | Net P/L: ${pnl:.4f} ({100*pnl/STARTING_CASH:.2f}%)")
    if position: print(f"Open paper position preserved: {position['symbol']} (report is mark-to-market when price is available).")
    if os.path.exists(TRADES_FILE):
        trades=pd.read_csv(TRADES_FILE); wins=trades[trades.pnl_usd>0]; losses=trades[trades.pnl_usd<0]
        win_rate=100*len(wins)/len(trades) if len(trades) else 0
        profit_factor=wins.pnl_usd.sum()/abs(losses.pnl_usd.sum()) if len(losses) and abs(losses.pnl_usd.sum())>0 else np.nan
        print(f"Trades: {len(trades)} | Win rate: {win_rate:.1f}% | Profit factor: {profit_factor:.2f}")
    if os.path.exists(EQUITY_FILE):
        equity=pd.read_csv(EQUITY_FILE); equity["time"]=pd.to_datetime(equity.time,utc=True,errors="coerce")
        equity=equity.dropna(subset=["time","equity","drawdown_pct"])
        if len(equity):
            print(f"Maximum drawdown: {equity.drawdown_pct.min():.2f}%")
            fig,axes=plt.subplots(2,1,figsize=(11,7),sharex=True,gridspec_kw={"height_ratios":[2,1]})
            axes[0].plot(equity.time,equity.equity,color="#2783DE",lw=2,label="Equity")
            axes[0].axhline(STARTING_CASH,color="#7D7A75",lw=1,ls="--",label="Starting balance")
            axes[0].set_ylabel("Virtual USD"); axes[0].legend(frameon=False); axes[0].set_title("Crypto Sentinel Pro — Session Performance")
            axes[1].fill_between(equity.time,equity.drawdown_pct,0,color="#E56458",alpha=.25)
            axes[1].plot(equity.time,equity.drawdown_pct,color="#E56458",lw=1.5)
            axes[1].set_ylabel("Drawdown %"); axes[1].set_xlabel("UTC time")
            for axis in axes: axis.spines[["top","right"]].set_visible(False)
            plt.tight_layout(); plt.savefig(REPORT_PNG,dpi=160,bbox_inches="tight"); plt.show()
    print("Logs:",TRADES_FILE,SIGNALS_FILE,EQUITY_FILE,STATE_FILE,sep="\n- ")

run_sentinel()


## Fixed-version setup

1. Upload this notebook to Kaggle and enable Internet.
2. Create a fine-grained GitHub token restricted to `fahadumrani/trader_agent` with **Contents: Read and write**.
3. Add it under **Kaggle → Add-ons → Secrets** as `GITHUB_TOKEN`; never place the token in code.
4. Select **Run All**.

### What was fixed
- Correct GitHub API URL and automatic default-branch detection
- Large-file download fallback and retry-safe GitHub synchronization
- UTC-safe historical CSV merging
- No fake label on the latest candle; latest row is inference-only
- Fee-aware, three-candle learning target
- Tie-correct AUC calculation and accurate labeled-row gate
- Best learned-ready candidate selection
- Duplicate signal prevention, exact cooldown handling and immediate state checkpoints
- Mark-to-market final reporting and safe handling of repeated network errors

The agent remains paper-only. Keep repository secrets and financial credentials out of GitHub.
